In [3]:
import pandas as pd
import numpy as np
from scipy import stats
import os

# --- CONFIGURATION ---
input_file = '../Results/final_radiomics_merged.csv' 
output_file = '../Results/table_statistical_analysis.csv'

print("--- AUTOMATED STATISTICAL AUDIT ---")

if not os.path.exists(input_file):
    print(f"ERROR: File not found: {input_file}")
else:
    df = pd.read_csv(input_file)
    print(f"Data Loaded: {len(df)} patients")
    
    cohorts = df['Cohort'].unique()
    print(f"Cohorts found: {cohorts}")
    
    if len(cohorts) < 2:
        print("Error: Need at least 2 cohorts to compare.")
    else:
        group1_name = cohorts[0]
        group2_name = cohorts[1]
        print(f"\nComparing: '{group1_name}' vs. '{group2_name}'")

        group1 = df[df['Cohort'] == group1_name]
        group2 = df[df['Cohort'] == group2_name]

        features = [c for c in df.columns if 'original_' in c]
        print(f"Analyzing {len(features)} radiomic features...")

        results = []

        for feature in features:
            v1 = group1[feature].dropna()
            v2 = group2[feature].dropna()
            
            # Descriptive Stats
            mean1, sd1 = v1.mean(), v1.std()
            mean2, sd2 = v2.mean(), v2.std()
            
            # Normality Check
            try:
                if len(v1) > 3 and len(v2) > 3:
                    _, p_norm1 = stats.shapiro(v1)
                    _, p_norm2 = stats.shapiro(v2)
                    is_normal = (p_norm1 > 0.05) and (p_norm2 > 0.05)
                else:
                    is_normal = False
            except:
                is_normal = False 

            # Significance Testing
            p_value = 1.0
            test_used = ""
            
            if is_normal:
                statistic, p_value = stats.ttest_ind(v1, v2, equal_var=False)
                test_used = "T-Test (Welch)"
            else:
                statistic, p_value = stats.mannwhitneyu(v1, v2)
                test_used = "Mann-Whitney U"
                
            # Levene's Test
            try:
                lev_stat, lev_p = stats.levene(v1, v2)
            except:
                lev_p = 1.0

            # Fold Change
            fold_change = mean2 / (mean1 + 1e-9)

            # Append Result - FIXED COLUMN NAME KEYS
            results.append({
                'Feature': feature,
                f'Mean ({group1_name})': f"{mean1:.2f} ± {sd1:.2f}",
                f'Mean ({group2_name})': f"{mean2:.2f} ± {sd2:.2f}",
                'Fold Change': f"{fold_change:.2f}x",
                'P_Value': p_value,  # Changed from 'P-Value' to 'P_Value' to match sort
                'Test Used': test_used,
                'Levene Variance P-Val': lev_p,
                'Significant?': 'YES' if p_value < 0.05 else 'No'
            })

        # Save & Display
        df_results = pd.DataFrame(results)
        
        # Sort by P_Value (Underscore matches now)
        df_results = df_results.sort_values(by='P_Value')
        
        # Save to CSV
        df_results.to_csv(output_file, index=False)
        
        print("\n--- ANALYSIS COMPLETE ---")
        print(f"Top 5 Significant Differences:")
        display(df_results.head(5))
        print(f"\nFull statistical table saved to: {os.path.abspath(output_file)}")

--- AUTOMATED STATISTICAL AUDIT ---
Data Loaded: 443 patients
Cohorts found: ['Public (Western)' 'Local (Square Hospital)']

Comparing: 'Public (Western)' vs. 'Local (Square Hospital)'
Analyzing 107 radiomic features...

--- ANALYSIS COMPLETE ---
Top 5 Significant Differences:


,Feature,Mean (Public (Western)),Mean (Local (Square Hospital)),Fold Change,P_Value,Test Used,Levene Variance P-Val,Significant?
98,original_shape_Sphericity,0.40 ± 0.05,0.73 ± 0.05,1.83x,3.880804e-15,Mann-Whitney U,0.976726,YES
22,original_shape_Maximum2DDiameterColumn,305.92 ± 55.49,125.27 ± 30.45,0.41x,9.742845e-15,Mann-Whitney U,0.129988,YES
47,original_shape_Maximum3DDiameter,312.71 ± 52.93,147.43 ± 38.66,0.47x,9.875301e-15,Mann-Whitney U,0.647228,YES
76,original_shape_Maximum2DDiameterRow,291.82 ± 57.80,139.55 ± 35.68,0.48x,1.326204e-14,Mann-Whitney U,0.190187,YES
49,original_shape_SurfaceVolumeRatio,0.15 ± 0.05,0.08 ± 0.02,0.53x,2.478786e-14,Mann-Whitney U,0.053406,YES



Full statistical table saved to: d:\Thesis_Project\Results\table_statistical_analysis.csv
